# Haystack pipeline reference

This notebook is a small, executable reference for working with Haystack and this repository's `retrieval-components` package. It demonstrates two equivalent workflows:

1. construct and run a pipeline directly in Python;
2. reconstruct and run the same pipeline from a fully resolved YAML configuration.

The example uses explicit two-dimensional embeddings, so it is deterministic and does not download a model.

## 1. Environment and imports

Run this notebook from the repository environment (for example, the environment created by `uv sync --project packages/retrieval-components --extra dev`). The path setup below also makes the local source tree importable when the notebook server starts elsewhere inside the repository.

In [ ]:
from pathlib import Path
import sys


def find_repository_root(start: Path) -> Path:
    """Find this monorepo from the current working directory."""
    for candidate in (start, *start.parents):
        package = candidate / "packages" / "retrieval-components" / "pyproject.toml"
        if package.is_file():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


repository_root = find_repository_root(Path.cwd().resolve())
components_src = repository_root / "packages" / "retrieval-components" / "src"
if str(components_src) not in sys.path:
    sys.path.insert(0, str(components_src))

example_dir = repository_root / "artifacts" / "notebook_examples"
example_dir.mkdir(parents=True, exist_ok=True)
repository_root

In [ ]:
import yaml
from haystack import Document, Pipeline

from retrieval_components.indexing.persisted_in_memory_document_indexer import (
    PersistedInMemoryDocumentIndexer,
)
from retrieval_components.retrieval.persisted_in_memory_embedding_retriever import (
    PersistedInMemoryEmbeddingRetriever,
)

## 2. Create a tiny persisted index

`PersistedInMemoryDocumentIndexer` is a component from `retrieval-components`. Its write session accumulates batches and atomically publishes the index only when the session succeeds. In production, an embedder would create the document embeddings; here they are explicit to keep the notebook self-contained.

In [ ]:
index_path = example_dir / "reference_embedding_index.json"

documents = [
    Document(
        id="hydra",
        content="Hydra composes experiment configuration from reusable groups.",
        embedding=[1.0, 0.0],
    ),
    Document(
        id="haystack",
        content="Haystack pipelines connect retrieval components together.",
        embedding=[0.0, 1.0],
    ),
    Document(
        id="metrics",
        content="Retrieval experiments compare ranked documents with qrels.",
        embedding=[0.5, 0.5],
    ),
]

In [ ]:
indexer = PersistedInMemoryDocumentIndexer(
    output_path=str(index_path),
    similarity="cosine",
    overwrite=True,
)

async with indexer.write_session():
    index_result = indexer.run(documents)

index_result

## 3. Construct and run a pipeline in pure Python

A Haystack pipeline is assembled by naming components, adding them, and connecting compatible sockets. This minimal pipeline has one component, so the run input is addressed directly to `retriever`. Multi-component pipelines use `pipeline.connect("sender.output", "receiver.input")`.

In [ ]:
python_pipeline = Pipeline(
    metadata={"description": "Offline dense-retrieval reference"}
)
python_pipeline.add_component(
    "retriever",
    PersistedInMemoryEmbeddingRetriever(
        index_path=str(index_path),
        top_k=2,
        return_embedding=False,
    ),
)

python_pipeline

In [ ]:
query_inputs = {
    "retriever": {
        "query_embedding": [0.0, 1.0],
        "candidate_document_ids": [document.id for document in documents],
    }
}

python_result = python_pipeline.run(query_inputs)
[
    {"id": document.id, "score": document.score, "content": document.content}
    for document in python_result["retriever"]["documents"]
]

## 4. Load and run a fully resolved pipeline configuration

Haystack's serialized form contains concrete component import paths, constructor parameters, connections, metadata, and execution settings. A **fully resolved** file has no Hydra defaults or `${...}` interpolations left. Such a file is portable and can be reconstructed without knowing how its source configuration was composed.

First, serialize the Python pipeline to create a small resolved reference file.

In [ ]:
resolved_pipeline_path = example_dir / "resolved_reference_pipeline.yaml"
resolved_pipeline_config = python_pipeline.to_dict()

with resolved_pipeline_path.open("w", encoding="utf-8") as config_file:
    yaml.safe_dump(resolved_pipeline_config, config_file, sort_keys=False)

print(resolved_pipeline_path.read_text(encoding="utf-8"))

Load the YAML as ordinary data and let Haystack instantiate the declared components. Import paths in the YAML must be importable in the active environment. The reconstructed pipeline accepts the same run inputs as the Python-built pipeline.

In [ ]:
with resolved_pipeline_path.open(encoding="utf-8") as config_file:
    loaded_pipeline_config = yaml.safe_load(config_file)

configured_pipeline = Pipeline.from_dict(loaded_pipeline_config)
configured_result = configured_pipeline.run(query_inputs)

configured_ranking = [
    document.id for document in configured_result["retriever"]["documents"]
]
python_ranking = [
    document.id for document in python_result["retriever"]["documents"]
]

assert configured_ranking == python_ranking
configured_ranking

## 5. Loading a pipeline from a resolved run artifact

Retrieval-core run directories save a larger `resolved_config.yaml` containing dataset, paths, runtime, stage, and pipeline settings. Only the `pipeline` mapping belongs to Haystack:

```python
with Path("artifacts/runs/<stage>/<run-id>/resolved_config.yaml").open() as file:
    resolved_run = yaml.safe_load(file)

pipeline = Pipeline.from_dict(resolved_run["pipeline"])
```

Use the same pattern with `AsyncPipeline.from_dict(...)` when the caller will use `await pipeline.run_async(...)`. The repository's stage runners use an async pipeline so independent branches and concurrent inputs can make progress efficiently.

## 6. Practical checks and conventions

- Call `pipeline.to_dict()` to inspect the exact serialized contract before saving a configuration.
- Keep experiment composition in Hydra configs, but archive the resolved config with every run for reproducibility.
- Treat component names and socket names as an API: changing either can break run inputs or connections.
- Keep paths in a resolved config valid for the environment in which it will run. Absolute paths are convenient locally; paths relative to the project root are easier to move.
- Call `pipeline.warm_up()` before latency-sensitive work when components load models or indexes lazily. `Pipeline.run()` also warms components as needed.
- Prefer tiny explicit fixtures like this one for pipeline wiring tests; reserve model-backed smoke tests for integration coverage.